<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

💻 🚙 PID pour le contrôle de l’orientation

Le contrôleur est la partie décisionnelle d’un robot.

Les contrôleurs produisent des _commandes_, qui sont des signaux exécutés par les actionneurs du robot, dans le but d’atteindre un objectif donné. Les entrées du contrôleur sont généralement des signaux (à partir duquel nous pouvons déduire l'état) qui permettent de définir cet objectif.

Par exemple, dans le cas du Duckiebot, les signaux de commande (en sortie) sont la vitesse linéaire $v$ et la vitesse angulaire $\omega$ du robot (En supposant que nous disposions d'un modèle cinématique capable de les convertir en commandes pour les roues gauche et droite). Autrement dit, le contrôleur décide, à chaque instant, à quelle vitesse le Duckiebot doit se déplacer et dans quelle mesure il doit tourner à gauche ou à droite afin, par exemple, de rester dans sa voie.

Le contrôleur Proportionnel–Intégral–Dérivé (PID) est un exemple de contrôleur à rétroaction (_feedback controller_) dans lequel l’objectif est défini en fonction d’un ou de plusieurs signaux d’erreur de poursuite ($e_t$), qui constituent les entrées du contrôleur. Le contrôleur cherche à ramener l’erreur idéalement à zéro, de sorte que la sortie du système ($\hat{x}_t$) devient égal au signal de référence ($x_{ref}$). 

$$ e_t = x_{ref} - \hat{x}_t \rightarrow 0 \Rightarrow \hat{x}_t = x_{ref}$$

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid_control_diagram.jpg" alt="pid-loop-2" style="width: 300px;"/>
  </div>
</figure>

Considérons un Duckiebot roulant au milieu d’une route avec une vitesse linéaire constante ($\bar v$). Dans la tâche de suivi de voie, notre objectif est de nous assurer que le robot reste au centre de la voie en ajustant sa vitesse angulaire. Intuitivement, chaque fois que le robot s’écarte du centre de la voie, nous ajustons la vitesse angulaire afin qu’il se dirige de nouveau vers le centre. La question est alors : de combien faut-il l’ajuster ?

Le terme proportionnel–intégral–dérivé provient du fait que le contrôleur ajuste les signaux de commande proportionnellement à l’erreur à chaque instant, à l’intégrale (c’est-à-dire la somme ou l’accumulation) de l’erreur au cours du temps, ainsi qu’à la dérivée de l’erreur à chaque pas de temps (c’est-à-dire au taux de variation de l’erreur dans le temps). 

La commande de contrôle est calculée en considérant la combinaison de ces trois composantes :

$$ u_t = k_p e(t) + k_i \int_0^t e(\tau) d \tau + k_d \frac{d e_t}{dt},$$

où $k_p$, $k_i$ et $k_d$ sont respectivement les coefficients proportionnel, intégral et dérivé du contrôleur.

Le problème du contrôle PID consiste à déterminer des valeurs pour ces paramètres (par exemple par essais et erreurs) jusqu’à ce que le système en boucle fermée présente des performances satisfaisantes. En général (mais cela dépend fortement de la complexité du système) :

 - Augmenter $k_p$ réduit le temps nécessaire au système pour s’approcher de la référence (c’est-à-dire le temps de montée "_rise time_"), mais au risque de dépasser la valeur de référence. Un $k_p$ élevé conduit à un contrôle agressif.

 - Augmenter $k_d$ permet de réduire ce dépassement en empêchant le robot de se déplacer trop rapidement dans une direction qui diminue  (contrebalancera le terme proportionnel lorsque la valeur absolue de l'erreur diminue et agira de concert avec le terme proportionnel lorsque la valeur absolue de l'erreur augmente).

 - Augmenter $k_i$ aide à éliminer l’erreur en régime permanent (c’est-à-dire l’erreur résiduelle une fois que le système a convergé) et à compenser des perturbations externes imprévues pendant le fonctionnement.

Par exemple, on peut commencer par ajuster uniquement $k_p$ en gardant $k_i = 0$ et $k_d = 0$, jusqu’à ce que le contrôleur parvienne à atteindre la référence avec un léger dépassement. On peut ensuite fixer $k_p$ à cette valeur et commencer à ajuster $k_d$. Une fois qu’une bonne valeur de $k_d$ est trouvée — réduisant les oscillations sans trop ralentir la réponse du système — on peut procéder à l’ajustement de $k_i$ afin de corriger les erreurs en régime permanent.

Bien que cette approche de réglage du contrôleur PID puisse fonctionner en pratique, il n’existe aucune garantie que le robot sera stable (par exemple, il peut osciller autour de la consigne, voire diverger complètement par rapport à celle-ci).

La popularité du contrôle PID tient notamment au fait qu’aucune méthode formelle n’est nécessaire pour obtenir un résultat satisfaisant. Des méthodes empiriques ("_rule of thumb_") comme la méthode de [Ziegler–Nichols](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method) peuvent aider à régler un contrôleur PID, mais les essais et erreurs — ou la « synthèse par itérations » fonctionnent également !

Passons maintenant à la conception d’un contrôleur PID !

## Commençons !

Nous allons concevoir un contrôleur proportionnel, intégral et dérivé (PID) afin de réguler l’orientation ($\theta_t$) d’un Duckiebot, alors qu’il se déplace à une vitesse linéaire constante, $\bar v$ (que nous ne chercherons pas à réguler).

En référence au schéma de contrôle PID ci-dessus, nous considérons les éléments suivants :

 - $x_{ref} = \theta_{ref}$ : le signal de référence est un angle constant, exprimé en radians;

 - $\hat x_t = \hat \theta_t$ : la variable contrôlée est l’orientation du Duckiebot. Nous allons estimer l’orientation du Duckiebot à partir du modèle d’odométrie conçu dans [l’activité sur l’odométrie](https://github.com/IFT3345/lx-kinematics-odometry);

 - $u_t = [\bar v, \omega_t]^T$ : la sortie du contrôleur, et l’entrée du système (le plant), seront deux variables. La première est la vitesse linéaire du robot, que nous supposerons constante. La variable que nous contrôlerons est la vitesse angulaire ($\omega_t = \dot{\theta}_t = \frac{d \theta_t}{dt}$).

L’objectif de cette activité est de déterminer les valeurs de $k_p$, $k_i$ et $k_d$ de manière à obtenir de « bonnes » performances de suivi.

Avant de passer à l’implémentation, examinons comment les ordinateurs calculent réellement les intégrales et les dérivées.

### Calcul d'intégrales en temps discret

La théorie nous dit que l’un des signaux composant le contrôleur PID est proportionnel à l’intégrale de l’erreur au cours du temps :

$$ e_{int}(t) = \int_0^t e(\tau) d\tau $$

Une intégrale est une somme _infinie_ de morceaux _infinitésimaux_, un concept qui suppose la continuité du temps. La continuité signifie que, pour deux instants dans le temps auxquels on peut penser, aussi proches soient-ils l’un de l’autre, il existera toujours un nombre infini d’autres instants de temps entre eux.

Mais aucun ordinateur au monde, y compris celui qui fonctionne sur le Duckiebot, ne sait gérer l’infini ou l’infinitésimal.

Les ordinateurs peuvent en revanche gérer le fini (au lieu de l’infini ou de l’infinitésimal). Le temps, pour un robot, est une succession d’instants. Mais lorsque l’on demande à un ordinateur de considérer deux instants consécutifs, il n’y a rien entre les deux. Les ordinateurs ont une notion du temps discret, et non du temps continu.

La conséquence immédiate de cette limitation fondamentale des ordinateurs est que nous ne pouvons pas réellement calculer des intégrales.

Ce que nous pouvons faire, c’est leur faire calculer une somme finie afin d’approximer l’intégrale réelle.

$$ e_{int}(t) = \int_0^t e(\tau) d\tau \simeq \sum_{i=0}^{k} e_i \Delta t = (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t$$

Dans l’approximation ci-dessus, nous avons supposé, par souci de simplicité, que tous les instants de temps sont espacés de manière uniforme par un pas de temps constant $\Delta t$.

Comment implémenter alors cette composante intégrale sur nos Duckiebots ? Nous pouvons remarquer que :

$$ e_{int,k}= (e_0 + e_1 + \dots + e_{k-1} + e_k)\Delta t = (e_0 + e_1 + \dots + e_{k-1})\Delta t + e_k\Delta t = e_{int,k-1} + e_k \Delta t.$$

In [ ]:
# Exécutez et ne modifiez pas cette cellule magique.
# Elle permet d’assurer le bon fonctionnement de l’ensemble du notebook Jupyter, 
# en particulier l’importation des modifications apportées aux fonctions définies dans des fichiers autres que cet espace de travail.
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

# Comment calculer le terme intégral de l'erreur de suivi ?

e_int_last = 0 # Erreur intégrale précédente, initialisée à 0
k = 10 # Exemple d'intervalle de temps actuel
e = np.ones(k) # On suppose que l'erreur est constante à chaque instant (ce qui n'est pas le cas)
dt = 0.1 # Exemple de période d'échantillonnage (secondes)

# Initialisation de l'erreur
e_int_current = 0

for i in range (0, k):
    ei = e[i] # erreur à cet instant
    e_int_current = e_int_last + ei * dt # erreur intégrale à l'instant précédent plus le nouvel incrément
    e_int_last = e_int_current 
    
print(f"La somme finie de l'erreur de suivi est {e_int_current}")

### Calcul des dérivées en temps discret

Les dérivées sont l'outil mathématique permettant de mesurer les variations.

Les dérivées temporelles sont définies comme le rapport entre la différence d'une fonction, évaluée en deux instants _infinitésimalement_ proches, et l'intervalle de temps qui les sépare.

Autrement dit, les dérivées sont les limites des fonctions de rapport incrémentales.


$$ \dot e_t = \frac{de_t}{dt} = \lim_{dt \rightarrow 0} \frac{e_{t+dt}-e_t}{dt}$$

Un ordinateur ne peut pas calculer directement les dérivées, mais il peut effectuer des calculs de différences finies, qui les approchent. Il existe de nombreuses formulations des différences finies ; la plus simple est la méthode d'Euler. Elle consiste à approximer la dérivée par la différence entre l'évaluation courante et l'évaluation précédente de la fonction, divisée par le pas de temps.

$$\frac{de_t}{dt} \simeq e_{der,k} = \frac{e_k - e_{k-1}}{\Delta t}.$$

Essayons !

In [ ]:
e_der = [0.0] # Initialisation du terme de dérivée
for i in range (1, k): # Remarque : on commence à 1, car on ne peut pas calculer la dérivée au premier instant, puisqu'il nous faut deux données discrètes.
    e_current = e[i] # Erreur à l'instant présent
    e_previous = e[i-1] # Erreur à l'instant précédent
    e_der_ = (e_previous - e_current)/dt
    e_der.append(e_der_)


print(f"La différence finie de l'erreur de suivi est {e_der}")

Nous disposons désormais de tous les outils nécessaires pour écrire notre premier contrôleur PID !

Rappelons que l’objectif est de contrôler l'orientation du robot en calculant sa vitesse angulaire, la vitesse linéaire étant maintenue constante.

## Implémenter le contrôleur PID

Implémentez la fonction `HeadingControl' dans le fichier [pid_controller.py](../packages/solution/pid_controller.py).



### Test unitaire

Ce test unitaire vous permet de tester rapidement le contrôleur que vous avez défini. Il chargera les gains PID à partir de fichier [HEADING_GAINS.yaml](../packages/solution/HEADING_GAINS.yaml). 

Vous trouverez la définition de ce test dans la fonction [unit_test](../packages/tests/unit_test.py).

In [ ]:
sys.path.append('../packages/tests/')
from unit_test import UnitTestHeadingPID

from solution.pid_controller import PIDController

import numpy as np

#Ceci sert uniquement à des fins de test rapide ; vous pouvez essayer différentes valeurs de v_init et de R, L, 
# ou celles que vous avez déterminées précédemment. Essayez de modifier R et L pour le plaisir.
v_test = 0.2
R_test = 0.018 # m
baseline_test = 0.1 # m
gain_test = 0.6 
trim_test = 0 
theta_ref= np.deg2rad(50) # en rad
# Vérification de cohérence (ne représente pas fidèlement le comportement réel, étant donné que le modèle de mouvement est 
# supposé être parfaitement connu)
unit_test = UnitTestHeadingPID(R_test, baseline_test, v_test, theta_ref, gain_test, trim_test, PIDController)
unit_test.test()

## Tester le contrôleur PID

Avant d'utiliser votre contrôleur PID sur un véritable Duckiebot, ce qui peut entraîner des plantages, il est préférable de commencer par le simulateur. Explorez les limites de votre contrôleur et ajustez les paramètres pour un fonctionnement optimal.   

Lors du passage du simulateur à votre Duckiebot, deux points essentiels sont à prendre en compte :

1. Sécurité : assurez-vous d’utiliser votre Duckiebot dans un espace où il ne risque pas de tomber d’une table, de descendre des escaliers ou d’entrer en collision avec des objets fragiles.

2. Comportement différent : votre Duckiebot se comportera différemment du Duckiebot simulé. Le modèle matériel simulé ne peut pas reproduire exactement votre Duckiebot assemblé, et l’environnement physique introduit de nombreux types d’erreurs absents de la simulation.


### 💻 Tester le contrôleur en simulation

Suivez les instructions du fichier [README](../README.md) pour créer un robot virtuel, démarrer le Duckiematrix, compiler et exécuter votre code, et enfin exécuter le navigateur VNC.

Ouvrez VNC dans votre navigateur et cliquez sur l'icône `PID_Heading` sur votre desktop. L'interface suivante s'ouvrira (cela peut prendre environ 30 secondes, voire plus, selon les caractéristiques de votre ordinateur) :

- Une interface RVIZ préconfigurée : pour visualiser les performances.
- Un terminal.
- Une fenêtre d'interaction avec les champs `ref` et `v_0`, et les boutons `Envoyer les commandes` et `Arrêter` (`v_0` et la référence pour la vitesse, $\bar v$ en haut). 

Dans la fenêtre RVIZ, vous devriez voir ce que voit le robot. Rien ne devrait bouger. Vous verrez des données de débogage dans le terminal de votre ordinateur, à partir duquel vous avez lancé l'activité. 

Pour initialiser les tests de votre contrôleur, vous devrez saisir les valeurs de `ref` (en degrés) et de `v_0` (entre -1 et 1), puis cliquer sur « Envoyer les commandes ». Le contrôleur que vous avez conçu précédemment calculera alors la valeur de ω et le robot se mettra en marche.

Pour cet exercice, les valeurs de référence `ref` = 10 et `v_0` = 0,2 sont appropriées. Vous pouvez toutefois les modifier en cours d'exécution et observer le comportement de votre contrôleur PID. Si vous vous éloignez trop de la route, vous pouvez utiliser le joystick pour ramener manuellement votre robot sur le chemin.

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-sim-start.png" alt="pid-heading-sim-1" style="width: 500px;"/>
  </div>
</figure>

Vous pouvez à tout moment appuyer sur le bouton « Stop » pour arrêter votre Duckiebot. 

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-heading-ctrl-sim.png" alt="pid-heading-sim-2" style="width: 500px;"/>
  </div>
</figure>

Pour tester différentes solutions, modifiez la fonction `HeadingControl` que vous avez écrite, enregistrez le fichier, recompilez et relancez le programme.  

Pour mieux comprendre le fonctionnement du système, examinez [le ROS node](../packages/pid_controller/src/pid_controller_node.py) qui subscribe aux informations d'odométrie, appelle votre fonction, puis publie les commandes de contrôle spécifiées par votre contrôleur (Comme dans le [labo ROS basics](https://github.com/IFT3345/lx-ros-basics), nous publions directement sur le sujet `joystick_cmd` qui envoie des commandes pour contrôler le robot.).

### 🚙 Testez sur votre Duckiebot

Vous pouvez suivre une procédure similaire à celle décrite ci-dessus (et décrite dans le [README](../README.md)) pour tester votre code sur votre véritable Duckiebot.

**Remarque** : nous vous conseillons de commencer à très basse vitesse avec le Duckiebot physique.

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-heading-control-real.png" alt="pid-heading-real-1" style="width: 300px;"/>
  <p>L'image ci-dessus montre le contrôle de l'orientation du Duckiebot avec:
  
$$v_{0} = 1, \theta_{ref} = \{0^\circ, 90^\circ, 180^\circ, 270^\circ, 0^\circ\}$$
</p>
  </div>
</figure>


Comme précédemment, modifiez le `HeadingControl` que vous avez écrit, enregistrez le fichier, puis reconstruisez et relancez-le.

**Remarque** : il est peu probable que vous puissiez obtenir les mêmes performances que celles illustrées dans la figure ci-dessus avec votre véritable Duckiebot.

Démontrez le comportement de votre Duckiebot à l'assistant de laboratoire.

### 💡 Réflexion

Après avoir testé votre contrôleur PID, en explorant différents coefficients PID et en faisant varier la vitesse et la distance du Duckiebot par rapport aux trajectoires de référence, vous avez peut-être constaté plusieurs points :

- Nécessité d'un réglage : La stabilité, les performances, les coefficients et l'amplitude de l'erreur de suivi sont liés. Un ensemble de coefficients performants pour une vitesse linéaire et une distance données par rapport à la référence peut s'avérer moins efficace avec d'autres valeurs. La régulation PID peut donner de bons résultats, mais elle nécessite un réglage adapté à chaque cas.

- Saturation des entrées : Les robots réels présentent des saturations (par exemple, `omega_max` dans les paramètres cinématiques). Cela signifie qu'une régulation plus agressive, même ponctuelle, ne modifiera pas la réponse du système, en raison d'autres contraintes liées au monde réel. Ces saturations peuvent refléter des limitations matérielles ou des contraintes de sécurité générales mises en place pour prévenir tout incident. Lorsque les entrées saturent, les erreurs (en particulier l'erreur intégrale) peuvent continuer à s'accumuler, perturbant ainsi le fonctionnement du contrôleur PID. Pour cette raison, des mécanismes anti-emballement sont généralement implémentés afin d'éviter la saturation des entrées ou une accumulation excessive de l'erreur intégrale. Il vous faudra peut-être en implémenter une pour empêcher cette saturation du terme intégral.

- On n'obtient jamais un suivi parfait, même avec un contrôleur PID très bien réglé. Quelles peuvent être les explications ?

- Le robot n'a aucune idée de ce qui se passe autour de lui. En effet, nous utilisons des capteurs intéroceptifs (encodeurs de roue) pour créer sa perception de son environnement. Son état réel lui est inconnu. De plus, l'absence de capteurs extéroceptifs (par exemple, une caméra) l'empêche d'ancrer sa perception à un élément du monde réel et d'effectuer des vérifications de cohérence. Vous avez peut-être remarqué qu'un Duckiebot ne prendra pas ce virage dans votre Duckietown de manière autonome, car il n'a aucune notion de carte. C'est pourquoi nous introduirons des caméras et la vision par ordinateur dans les modules futurs.